In [47]:
from lakehouse.daft import bronze, silver
from lakehouse.daft.utils import daftutils
import daft
import datetime as dt

In [48]:
CATALOG = "daft_catalog"

# 1. Set Up and Bronze Data

In [49]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [50]:
class TestBronze(bronze.Bronze):
    def custom_load(self, table):
        #creates a df with column id range 0-9
        df = daftutils.idrange(10)
        df = df.with_column("t", daft.lit(table))
        return df
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"


bronze_instance = TestBronze(**options)

In [51]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-03-15 22:27:55 | people | execute | Started
2025-03-15 22:27:55 | people | load | Started
2025-03-15 22:27:55 | people | load | Completed in 0.0 min
2025-03-15 22:27:55 | people | transform | Started
2025-03-15 22:27:55 | people | transform | Completed in 0.0 min
2025-03-15 22:27:55 | people | write | Started
2025-03-15 22:27:55 | people | write | Completed in 0.0 min
2025-03-15 22:27:55 | people | execute | Completed in 0.0 min
2025-03-15 22:27:55 | planets | execute | Started
2025-03-15 22:27:55 | planets | load | Started
2025-03-15 22:27:55 | planets | load | Completed in 0.0 min
2025-03-15 22:27:55 | planets | transform | Started
2025-03-15 22:27:55 | planets | transform | Completed in 0.0 min
2025-03-15 22:27:55 | planets | write | Started
2025-03-15 22:27:55 | planets | write | Completed in 0.0 min
2025-03-15 22:27:55 | planets | execute | Completed in 0.0 min


In [52]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",idInt64,tUtf8
2025-03-15 22:27:55.482274,0,people
2025-03-15 22:27:55.482274,1,people
2025-03-15 22:27:55.482274,2,people
2025-03-15 22:27:55.482274,3,people
2025-03-15 22:27:55.482274,4,people
2025-03-15 22:27:55.482274,5,people
2025-03-15 22:27:55.482274,6,people
2025-03-15 22:27:55.482274,7,people


No. Rows: 10


In [53]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Debug with Overwrite example

In [54]:
class TestSilver(silver.Silver):
    def custom_filter(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.where("id <= 5")

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.with_column("col", daft.lit("col"))
    
    def source_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.source_schema}/{table}"
    
    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"


silver_instance = TestSilver(**options)

## 2.1 Debug the data load

In [55]:
# Debug without filter
silver_instance.load().execute("people")
actual_df = silver_instance.data["people"]
current_timestamp = dt.datetime.now()
expected_df = daftutils.idrange(10).with_column("t", daft.lit("people")).with_column("LH_BronzeTS", daft.lit(current_timestamp)).select("LH_BronzeTS", "id", "t")
actual_df.show()
daftutils.assert_schema_equal(actual_df.schema(), expected_df.schema())
daftutils.assert_frame_equal(actual_df.exclude("LH_BronzeTS"), expected_df.exclude("LH_BronzeTS"))

"LH_BronzeTSTimestamp(Microseconds, None)",idInt64,tUtf8
2025-03-15 22:27:55.482274,0,people
2025-03-15 22:27:55.482274,1,people
2025-03-15 22:27:55.482274,2,people
2025-03-15 22:27:55.482274,3,people
2025-03-15 22:27:55.482274,4,people
2025-03-15 22:27:55.482274,5,people
2025-03-15 22:27:55.482274,6,people
2025-03-15 22:27:55.482274,7,people


In [56]:
# Debug with filter
silver_instance.load(filter="custom").execute("people")
actual_df = silver_instance.data["people"]
expected_df = daftutils.idrange(10).with_column("t", daft.lit("people")).with_column("LH_BronzeTS", daft.lit(current_timestamp)).where("id <= 5").select("LH_BronzeTS", "id", "t")
actual_df.show()
daftutils.assert_schema_equal(actual_df.schema(), expected_df.schema())
daftutils.assert_frame_equal(actual_df.exclude("LH_BronzeTS"), expected_df.exclude("LH_BronzeTS"))

"LH_BronzeTSTimestamp(Microseconds, None)",idInt64,tUtf8
2025-03-15 22:27:55.482274,0,people
2025-03-15 22:27:55.482274,1,people
2025-03-15 22:27:55.482274,2,people
2025-03-15 22:27:55.482274,3,people
2025-03-15 22:27:55.482274,4,people
2025-03-15 22:27:55.482274,5,people


# 2.2 Debug transformation

In [57]:
# Debug with default transformation
silver_instance.load(filter="custom").transform().execute("people")
actual_df = silver_instance.data["people"]
expected_df = daftutils.idrange(10).with_column("t", daft.lit("people")).with_column("LH_BronzeTS", daft.lit(current_timestamp)).where("id <= 5").with_column("col", daft.lit("col")).with_column("LH_SilverTS", daft.lit(current_timestamp)).select("LH_SilverTS", "LH_BronzeTS", "id", "t", "col")
actual_df.show()
daftutils.assert_schema_equal(actual_df.schema(), expected_df.schema())
daftutils.assert_frame_equal(actual_df.exclude("LH_BronzeTS", "LH_SilverTS"), expected_df.exclude("LH_BronzeTS", "LH_SilverTS"))

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",idInt64,tUtf8,colUtf8
2025-03-15 22:27:57.221569,2025-03-15 22:27:55.482274,0,people,col
2025-03-15 22:27:57.221569,2025-03-15 22:27:55.482274,1,people,col
2025-03-15 22:27:57.221569,2025-03-15 22:27:55.482274,2,people,col
2025-03-15 22:27:57.221569,2025-03-15 22:27:55.482274,3,people,col
2025-03-15 22:27:57.221569,2025-03-15 22:27:55.482274,4,people,col
2025-03-15 22:27:57.221569,2025-03-15 22:27:55.482274,5,people,col


In [58]:
# Debug without default transformation
silver_instance.load(filter="custom").transform(ignore_defaults=True).execute("people")
actual_df = silver_instance.data["people"]
expected_df = daftutils.idrange(10).with_column("t", daft.lit("people")).with_column("LH_BronzeTS", daft.lit(current_timestamp)).where("id <= 5").with_column("col", daft.lit("col")).select("LH_BronzeTS", "id", "t", "col")
actual_df.show()
daftutils.assert_schema_equal(actual_df.schema(), expected_df.schema())
daftutils.assert_frame_equal(actual_df.exclude("LH_BronzeTS"), expected_df.exclude("LH_BronzeTS"))

"LH_BronzeTSTimestamp(Microseconds, None)",idInt64,tUtf8,colUtf8
2025-03-15 22:27:55.482274,0,people,col
2025-03-15 22:27:55.482274,1,people,col
2025-03-15 22:27:55.482274,2,people,col
2025-03-15 22:27:55.482274,3,people,col
2025-03-15 22:27:55.482274,4,people,col
2025-03-15 22:27:55.482274,5,people,col


# 2.3 Debug write

In [59]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute(
    "people"
)
actual_df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/people")
expected_df = daftutils.idrange(10).with_column("t", daft.lit("people")).with_column("LH_BronzeTS", daft.lit(current_timestamp)).where("id <= 5").with_column("col", daft.lit("col")).with_column("LH_SilverTS", daft.lit(current_timestamp)).select("LH_SilverTS", "LH_BronzeTS", "id", "t", "col")
actual_df.show()
daftutils.assert_schema_equal(actual_df.schema(), expected_df.schema())
daftutils.assert_frame_equal(actual_df.exclude("LH_BronzeTS", "LH_SilverTS"), expected_df.exclude("LH_BronzeTS", "LH_SilverTS"))

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",idInt64,tUtf8,colUtf8
2025-03-15 22:27:58.475146,2025-03-15 22:27:55.482274,0,people,col
2025-03-15 22:27:58.475146,2025-03-15 22:27:55.482274,1,people,col
2025-03-15 22:27:58.475146,2025-03-15 22:27:55.482274,2,people,col
2025-03-15 22:27:58.475146,2025-03-15 22:27:55.482274,3,people,col
2025-03-15 22:27:58.475146,2025-03-15 22:27:55.482274,4,people,col
2025-03-15 22:27:58.475146,2025-03-15 22:27:55.482274,5,people,col


# 3 Clean Up

In [60]:
import shutil
shutil.rmtree(f"D:/Data/{CATALOG}")